In [ ]:
import os

DatasetRoot = '../data/Derived_Features'
feat_categories_names = sorted(dir_name for dir_name in os.listdir(DatasetRoot))
assert len(feat_categories_names) == 5, feat_categories_names

feat_cats = [' '.join(f.split('_')) for f in feat_categories_names]
print(*feat_cats, sep='\n')

In [ ]:
DynamicFeature = "Dynamic Feature"
StaticCodeStructureFeature = "Static Code Structure Feature"
StaticCodeGraphFeature = "Static_Code_Graph_Feature"
GraphCodeBertFeature = "Static_Code_GraphCodeBert_Feature"

GraphCodeBert Feature

In [ ]:
def prepare_graph_codebert_dataset(verbose=True):
    import pandas as pd
    import json
    
    from sklearn.preprocessing import LabelEncoder

    model_files = os.listdir(os.path.join(DatasetRoot, GraphCodeBertFeature))
    _model_names = [model_file.split('_')[1] for model_file in model_files]
    _labels = [model_file.split('_')[2] for model_file in model_files]
    
    if verbose:
        print('Number of models:', len(_model_names))
        print('Number of unique labels:', len(set(_labels)))
    
    X = []
    for model_file in model_files:
        df = pd.read_csv(os.path.join(DatasetRoot, GraphCodeBertFeature, model_file))
        embedding = df.iloc[0].embedding
        X.append(json.loads(embedding))
        
    assert len(X) == len(_model_names) == len(_labels)

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(_labels)
    
    if verbose:
        print('Labels:', *label_encoder.classes_)
    
    if verbose:
        print('Embedding shape:', len(X[0]))
    
    return X, y

X, y = prepare_graph_codebert_dataset()

Train Models

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13)

In [ ]:
# Try all models possible
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
# from xgboost import XGBClassifier
from tqdm import tqdm

def run_models(X_train, X_test, y_train, y_test, verbose=True):
    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000),
        'DecisionTree': DecisionTreeClassifier(),
        'RandomForest': RandomForestClassifier(),
        'SVC': SVC(max_iter=1000),
        'KNeighbors': KNeighborsClassifier(),
        'MLP': MLPClassifier(max_iter=1000),
        'GradientBoosting': GradientBoostingClassifier(),
        'AdaBoost': AdaBoostClassifier(),
        'Bagging': BaggingClassifier(),
        # 'XGBoost': XGBClassifier()
    }
    
    results = {}
    
    for model_name, model in tqdm(models.items(), desc='Training Models', total=len(models)):
        model.fit(X_train, y_train)
        results[model_name] = model.score(X_test, y_test)
        
        if verbose:
            print(model_name, results[model_name])
            
    return results

In [ ]:
results = run_models(X_train, X_test, y_train, y_test, verbose=False)
results

Graph Feature

In [ ]:
def prepare_static_code_graph_dataset(verbose=True):
    import pandas as pd
    import json
    
    from sklearn.preprocessing import LabelEncoder

    model_files = os.listdir(os.path.join(DatasetRoot, StaticCodeGraphFeature))
    _model_names = [model_file.split('_')[1] for model_file in model_files]
    _labels = [model_file.split('_')[2] for model_file in model_files]
    
    if verbose:
        print('Number of models:', len(_model_names))
        print('Number of unique labels:', len(set(_labels)))
    
    X_mean = []
    X_median = []
    X_mode = []
    X_max = []
    cols = None
    for i, model_file in enumerate(model_files):
        df = pd.read_csv(os.path.join(DatasetRoot, StaticCodeGraphFeature, model_file))
        if cols is None:
            cols = [col for col in df.columns if col != 'Shortest_Path_Length']
        feature_dicts: dict[str, dict[int, float]] = {}
        for col in cols:
            if col in ['Transitivity', 'Density']:
                feature_dicts[col] = {0: float(df[col].iloc[0])}
                continue
            try:
                feature_dicts[col] = eval(df[col].iloc[0])
            except NameError as e:
                # This happens when we encounter a NaN or Inf value
                # We will replace them with 0
                feature_dicts[col] = {0: 0}
            except SyntaxError as e:
                print(f'{i}. {model_file}')
                print('col =', col)
                print('value =', df[col].iloc[0])
                print('SyntaxError:', e)
                raise e
        
        features = [
            list(feature_dicts[col].values()) for col in cols
        ]
            
        X_mean.append([sum(f)/len(f) for f in features])
        X_median.append([sorted(f)[len(f)//2] for f in features])
        X_mode.append([max(set(f), key=f.count) for f in features])
        X_max.append([max(f) for f in features])
        
    assert len(X_mean) == len(_model_names) == len(_labels)

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(_labels)
    
    if verbose:
        print('Columns:', *cols, sep=', ')
        print(f'X.shape: ({len(X_mean)}, {len(X_mean[0])})')
        print('Labels:', *label_encoder.classes_)
        
    return X_mean, X_median, X_mode, X_max, y


X_mean, X_median, X_mode, X_max, y = prepare_static_code_graph_dataset()

In [ ]:
from sklearn.model_selection import train_test_split

X_train_mean, X_test_mean, \
    X_train_median, X_test_median, \
    X_train_mode, X_test_mode, \
    X_train_max, X_test_max, \
    y_train, y_test = \
    train_test_split(X_mean, X_median, X_mode, X_max, y, test_size=0.2, random_state=13)

In [ ]:
graph_datasets = {
    'Mean': (X_train_mean, X_test_mean),
    'Median': (X_train_median, X_test_median),
    'Mode': (X_train_mode, X_test_mode),
    'Max': (X_train_max, X_test_max)
}

def run_graph_models(graph_datasets, y_train, y_test, verbose=True):
    results = {}
    
    for dataset_name, (X_train, X_test) in graph_datasets.items():
        results[dataset_name] = run_models(X_train, X_test, y_train, y_test, verbose)
        
    return results

In [ ]:
results = run_graph_models(graph_datasets, y_train, y_test, verbose=False)

In [ ]:
from pprint import pprint

for dataset_name, result in results.items():
    print(dataset_name)
    pprint(result)
    
print('Max:', max(max(result.values()) for result in results.values()))

Local Code Structure

In [ ]:
import os

os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import pandas as pd
import keras

In [ ]:
DatasetRoot = '../data/Derived_Features/Static_Code_Structure_Feature'

In [ ]:
def read_dataset(verbose=False):
    from tqdm import tqdm

    filenames = os.listdir(DatasetRoot)
    NumEntries = len(filenames)
    NumLayers = 15
    NumFeatures = 12
    raw_data = None

    if verbose:
        print(f'Number of entries: {NumEntries}')

    labels = [
        filename.split('_')[2]
        for filename in filenames
    ]

    if verbose:
        print('Unique labels:', set(labels))

    for i, filename in enumerate(tqdm(filenames)):
        df = pd.read_csv(
            os.path.join(DatasetRoot, filename),
            keep_default_na=False,
        ).drop(
            columns=['Sequential_Index', 'input_shape', 'output_shape', 'label'],
        )

        if 'units' in df.columns:
            df = df.drop(columns=['units'])

        df['use_bias'] = (df['use_bias'] == 'True')
        df['trainable'] = (df['trainable'] == 'True')
        df['is_reshaping_layer'] = (df['is_reshaping_layer'] != 'False')
        weights = df['weights_shape'].apply(lambda w: eval(w) if w else tuple())
        biases = df['biases_shape'].apply(lambda b: eval(b) if b else tuple())
        df = df.drop(columns=['weights_shape', 'biases_shape'])
        assert (weights.str.len() <= 4).all(), f'{filename} {weights.str.len()}'
        assert (biases.str.len() <= 1).all(), f'{filename} {biases.str.len()}'

        df['input_shape'] = [w[0] if w else 0 for w in weights]
        df['mejo_shape'] = [w[1] if len(w) == 3 else (1 if w else 0) for w in weights]
        df['sejo_shape'] = [w[2] if len(w) == 4 else (1 if w else 0) for w in weights]
        df['output_shape'] = [w[-1] if w else 0 for w in weights]
        df['bias_shape'] = [b[0] if b else 0 for b in biases]

        # pad dataset rows to match NumLayers
        while len(df) < NumLayers:
            df.loc[len(df)] = ['', '', '', '', False, False, False, 0, 0, 0, 0, 0]

        if raw_data is None:
            raw_data = df.values[None, :]
        else:
            raw_data = np.concatenate([raw_data, df.values[None, :]], axis=0)

    if verbose:
        print(f'Raw data shape: {raw_data.shape}')
        print('Sample data:')
        print(raw_data[0, 1])

    return raw_data, labels


X_raw, y_raw = read_dataset(verbose=True)

In [ ]:
def prepare_dataset(X_raw, y_raw, verbose=False):
    from sklearn.preprocessing import OneHotEncoder

    if verbose:
        print(f'Raw data shape: {X_raw.shape}')

    preprocessed_data = X_raw[:, :, 4:].astype(float)  # from the 5th layer all are numerical features

    for i in range(4):
        encoder = OneHotEncoder()
        to_be_encoded = X_raw[:, :, i]
        linearized_data = to_be_encoded.reshape(-1, 1)
        encoder.fit(linearized_data)
        encoded = encoder.transform(linearized_data).toarray().reshape(X_raw.shape[0], X_raw.shape[1], -1)
        if verbose:
            print(f'{i}. input shape: {to_be_encoded.shape}')
            print(f'{i}. output shape: {encoded.shape}')
            print(f'{i}. category:', *encoder.categories_)
            print()
        preprocessed_data = np.concatenate([
            encoded,
            preprocessed_data,
        ], axis=2)

    output_encoder = OneHotEncoder()
    reshaped_output = [[label] for label in y_raw]
    output_encoder.fit(reshaped_output)
    y = output_encoder.transform(reshaped_output).toarray().reshape(-1, len(output_encoder.categories_[0]))

    if verbose:
        print('output shape:', y.shape)
        print('output category:', *output_encoder.categories_)
        print(f'Preprocessed data shape: {preprocessed_data.shape}')
        print('Sample data:')
        print(preprocessed_data[111, 11])

    return preprocessed_data, y


X, y = prepare_dataset(X_raw, y_raw, verbose=True)

In [ ]:
def split_and_scale_dataset(X, y, verbose=False):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
    X_test = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)
    
    if verbose:
        print('X_train:', X_train.shape)
        print('X_test:', X_test.shape)
        print('y_train:', y_train.shape)
        print('y_test:', y_test.shape)
        
        print('scaler mean:', scaler.mean_)
        print('scaler var:', scaler.var_)

    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = split_and_scale_dataset(X, y, verbose=True)

Build, Train and Evaluate Model

In [ ]:
from keras import Model
from keras.api.layers import Input, Dense, Conv1D, Conv2D, MaxPooling1D, MaxPooling2D, Flatten, concatenate


def build_model(input_shape, num_classes):
    input_layer = Input(shape=input_shape, name='input')

    conv1 = Conv1D(32, kernel_size=(3,), activation='relu', name='conv1')(input_layer)
    conv2 = Conv1D(64, kernel_size=(3,), activation='relu', name='conv2')(conv1)
    pool1 = MaxPooling1D(pool_size=(2,), name='pool1')(conv2)
    flatten = Flatten(name='flatten')(pool1)
    dense1 = Dense(128, activation='relu', name='dense')(flatten)
    output_layer = Dense(num_classes, activation='softmax', name='output')(dense1)

    model = Model(input_layer, output_layer, name='static_model')
    model.compile(
        loss=keras.losses.CategoricalCrossentropy(),
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        metrics=[
            keras.metrics.CategoricalAccuracy(name="acc"),
        ],
    )

    return model

In [ ]:
model = build_model(X_train.shape[1:], y_train.shape[1])
model.summary()

In [ ]:
from keras.api.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_acc', patience=5)

history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=100,
    verbose=1,
    validation_data=(X_train, y_train),
    callbacks=[early_stopping],
)

if not os.path.exists('../models'):
    os.makedirs('../models')

model.save('../models/static_model.keras')

In [ ]:
def evaluate_model(model, X_test, y_test):
    from sklearn.metrics import classification_report

    y_pred = model.predict(X_test)
    y_pred = y_pred.argmax(axis=1)
    y_true = y_test.argmax(axis=1)

    print(classification_report(y_true, y_pred))
    
evaluate_model(model, X_test, y_test)

In [ ]:
import seaborn as sns

history_df = pd.DataFrame(history.history)
history_df['acc'] = history_df['acc'] * 100
sns.lineplot(data=history_df[['acc', 'loss']])

In [ ]:
history.history